# PDelta3 / GDN2 on arnir0/Tiny-LLM

This notebook checks whether the recurrent attention ideas that worked best in SmolLM2 also transfer to **arnir0/Tiny-LLM**.

Tiny-LLM has only **one Llama block**, so literal cross-layer value routing (CLVR) cannot exist. We therefore test:

1. `conv4_pdelta_f96` — current bounded-state control;
2. `conv4_channel_decay_f96` — stronger content-dependent channel decay;
3. `conv4_gdn2_f96` — independent erase/write GDN2 recurrence;
4. `conv4_gdn2_inputroute_f96` — a **one-layer value-routing proxy** that routes current pre-convolution V through the CLVR projection/gate. This is explicitly **not** true CLVR.

The host Transformer attention remains the baseline. Architecture selection uses validation only, while the held-out test set is evaluated once at the end with paired bootstrap confidence intervals.


In [ ]:
import os, sys, subprocess, tempfile
from pathlib import Path

assert subprocess.run(["nvidia-smi"], check=False).returncode == 0, "Enable a GPU runtime in Colab."
REPO = Path(tempfile.mkdtemp(prefix="TinyCeNN-tinyllm-"))
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "datasets", "huggingface_hub", "pandas", "matplotlib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "--no-deps"], check=True)
print("Repository:", REPO)
subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True)


In [ ]:
from datetime import datetime, timezone

PROFILE = "balanced"          # quick | balanced | strong
CURRICULUM = "256,512,1024"
VALIDATION_CONTEXTS = "256,512,1024"
TEST_CONTEXTS = "256,512,1024"
SEED = 2026

RESULT_ROOT = Path("/content/TinyCeNN-tinyllm-results")
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = RESULT_ROOT / f"{PROFILE}-{stamp}"
print("Output:", OUTPUT_DIR)


In [ ]:
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained("arnir0/Tiny-LLM", revision="b784a70a5e6908c9148820a245d60a3347279868")
print({
    "layers": cfg.num_hidden_layers,
    "hidden_size": cfg.hidden_size,
    "heads": cfg.num_attention_heads,
    "kv_heads": cfg.num_key_value_heads,
    "head_dim": cfg.hidden_size // cfg.num_attention_heads,
    "max_context": cfg.max_position_embeddings,
})
assert cfg.num_hidden_layers == 1
print("True CLVR is unavailable because Tiny-LLM has no preceding Transformer layer.")


In [ ]:
cmd = [
    sys.executable,
    str(REPO / "scripts" / "benchmark_tiny_llm_pdelta3_colab.py"),
    "--profile", PROFILE,
    "--curriculum", CURRICULUM,
    "--validation-contexts", VALIDATION_CONTEXTS,
    "--test-contexts", TEST_CONTEXTS,
    "--seed", str(SEED),
    "--output-dir", str(OUTPUT_DIR),
]
print(" ".join(cmd), flush=True)
process = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, cmd)


In [ ]:
import json
import pandas as pd
from IPython.display import display

validation = pd.read_csv(OUTPUT_DIR / "validation_context_summary.csv")
test = pd.read_csv(OUTPUT_DIR / "test_summary.csv")
diag = pd.read_csv(OUTPUT_DIR / "candidate_diagnostics.csv")
selection = json.loads((OUTPUT_DIR / "selection.json").read_text())
report = json.loads((OUTPUT_DIR / "tiny_llm_pdelta3_report.json").read_text())

print("SELECTION")
print(json.dumps(selection, indent=2))
print("\nVALIDATION")
display(validation.sort_values(["context", "delta_nll"]))
print("\nHELD-OUT TEST")
display(test.sort_values(["context", "delta_nll"]))
print("\nDIAGNOSTICS")
display(diag.sort_values("prefill_ms"))
strict = test[test["verdict"] == "strict_quality_win"]
print("\nSTRICT TRANSFORMER QUALITY WIN:", "YES" if len(strict) else "NO")
if len(strict):
    display(strict)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
for name, part in test.groupby("candidate"):
    part = part.sort_values("context")
    plt.plot(part["context"], part["delta_nll"], marker="o", label=name)
plt.axhline(0.0, linewidth=1)
plt.axhline(0.02, linewidth=1, linestyle="--")
plt.xlabel("Context length")
plt.ylabel("Candidate - Transformer NLL")
plt.title("Tiny-LLM: recurrent replacement quality gap")
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(9, 5))
for name, part in test.groupby("candidate"):
    part = part.sort_values("context")
    plt.plot(part["context"], 100 * part["state_vs_transformer_fp16"], marker="o", label=name)
plt.xlabel("Context length")
plt.ylabel("Persistent state / Transformer FP16 KV (%)")
plt.title("Tiny-LLM persistent-state scaling")
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Downloading complete experiment:", archive)
files.download(archive)
